In [15]:
!pip install -U langchain langchain-core langchain-community langchain-google-genai streamlit pyngrok bs4 pypdf chromadb faiss-cpu sentence-transformers langchain_classic

In [2]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

print("Downloading llama2 model (this may take a few minutes)...")
!ollama pull llama2
print("Done! Ollama is now ready to serve queries.")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [82.9 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.6 MB]
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [55.6 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:13 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2,277 kB

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/attention.pdf")

docs = loader.load()
docs

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 20
)

documents = text_splitter.split_documents(docs)

documents

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = Chroma.from_documents(documents, embeddings)
print("Chroma Vector Store created successfully using local HuggingFace Embeddings!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chroma Vector Store created successfully using local HuggingFace Embeddings!


In [12]:
from langchain_community.llms import Ollama

llm = Ollama(model = "llama2")
llm

/tmp/ipykernel_3299/1506236888.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model = "llama2")


Ollama(metadata={'lc_versions': {'langchain-core': '1.8.5', 'langchain': '1.5.2'}})

In [13]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context.
Think step by step before providing a detailed answer.
I will tip you $1000 if the user finds the answer helpful.
<context>
{context}
</context>
Question: {input}
""")

##### Chains

A chain in LangChain is a composable pipeline that connects multiple components such as prompts, large language models (LLMs), output parsers, and external tools into a single, executable workflow where the output of one step becomes the input for the next.

In [19]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm, prompt)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context. \nThink step by step before providing a detailed answer. \nI will tip you $1000 if the user finds the answer helpful. \n<context>\n{context}\n</context>\nQuestion: {input}\n'), additional_kwargs={})])
| Ollama(metadata={'lc_versions': {'langchain-core': '1.8.5', 'langchain': '1.5.2'}})
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

##### Retrievers:
A retrievers is an interface that returns documents given an unstructured query.

In [18]:
retrievers = db.as_retriever()

retrievers

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7a17a5fbec10>, search_kwargs={})

##### Retrievers Chain:
This chain takes in a user inquiry, which is then passed to the retrievers to fetch relevant documents.

In [21]:
from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retrievers, document_chain)

In [30]:
# Start Ollama server in background and wait for it
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Ensure model is ready and invoke chain
subprocess.run(["ollama", "pull", "llama2"], capture_output=True)

response = retrieval_chain.invoke({"input": "An attention function can be described as mapping a query"})
response['answer']

'The attention function can be described as mapping a query to an output by computing a weighted sum of values, where the weights are computed based on the compatibility between the query and the corresponding keys. The attention function can be implemented using two main types of attention mechanisms:\n\n1. Scaled Dot-Product Attention: This attention mechanism computes the weights by taking the dot product of the query and keys, scaling the results by 1√dk, and applying a softmax function to obtain the weights.\n2. Multi-Head Attention: This attention mechanism splits the queries, keys, and values into multiple parallel attention layers, each with its own learned linear projection. The outputs of each layer are then concatenated and linearly transformed to obtain the final output.\n\nThe choice of attention mechanism depends on the specific task and the size of the input data. In general, dot-product attention is faster and more space-efﬁcient in practice, while additive attention pr